
# Day 4 — Linear Probing Evaluation

## 实验目标

评估 MoCo 自监督预训练所学习的视觉表征质量。

### 方法

冻结 ResNet-18 Backbone，仅训练一个线性分类器：

$$
h = f_{\theta}(x) \in \mathbb{R}^{512}
$$

$$
\hat{y} = Wh+b
$$

其中：

- $f_{\theta}$：冻结的 ResNet-18 Backbone
- $W,b$：可训练的线性分类器参数
- $\hat{y}$：10个 CIFAR-10 类别的分类 logits

### 对照实验

1. MoCo Pretrained Backbone + Linear Probe
2. Randomly Initialized Backbone + Linear Probe

### 评估指标

- Test Accuracy
- Macro F1-Score
- Confusion Matrix

整个 Linear Probe 训练过程中，Backbone 参数保持冻结。


In [ ]:
# 加载Moco Checkpoint
import sys
from pathlib import Path

import torch
import torch.nn as nn

cwd = Path.cwd().resolve()

project_root = (
    cwd if (cwd / "src").is_dir()
    else cwd.parent
)

sys.path.insert(0, str(project_root))

from src.moco import ContrastiveEncoder, MoCo

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

checkpoint_path = (
    project_root / "checkpoints" / "moco_latest.pth"
)

assert checkpoint_path.exists(), "Checkpoint not found!"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

config = checkpoint["config"]

moco = MoCo(
    feature_dim=config["feature_dim"],
    queue_size=config["queue_size"],
    momentum=config["momentum"],
    temperature=config["temperature"]
)

moco.load_state_dict(checkpoint["model_state_dict"])

print("Checkpoint epoch:", checkpoint["epoch"])
print("Checkpoint loaded successfully!")


Checkpoint epoch: 5
Checkpoint loaded successfully!


In [2]:
# 提取冻结的Backbone

import copy

# Pretrained backbone
pretrained_backbone = copy.deepcopy(
    moco.encoder_q.backbone
).to(device)

# Random baseline backbone
torch.manual_seed(123)

random_encoder = ContrastiveEncoder(
    feature_dim=config["feature_dim"]
)

random_backbone = random_encoder.backbone.to(device)

# Freeze both backbones
for backbone in [pretrained_backbone, random_backbone]:
    backbone.eval()

    for param in backbone.parameters():
        param.requires_grad = False

print("Pretrained backbone frozen.")
print("Random backbone frozen.")
print("Output feature dimension: 512")


Pretrained backbone frozen.
Random backbone frozen.
Output feature dimension: 512


In [3]:
# 加载分类数据集：预训练使用随机裁剪、ColorJitter 等增强来构造正样本对。Linear Probe 则需要保留清晰的类别标签，并使用确定性的图像预处理，保证公平比较
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

eval_transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.CIFAR10(
    root=str(project_root / "data"),
    train=True,
    download=False,
    transform=eval_transform
)

test_dataset = datasets.CIFAR10(
    root=str(project_root / "data"),
    train=False,
    download=False,
    transform=eval_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))
print("Classes:", train_dataset.classes)


Train samples: 50000
Test samples: 10000
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [4]:
#检查特征是否正确
images, labels = next(iter(train_loader))

images = images[:8].to(device)

with torch.no_grad():
    h_pretrained = pretrained_backbone(images)
    h_random = random_backbone(images)

print("Pretrained features:", h_pretrained.shape)
print("Random features:", h_random.shape)

assert h_pretrained.shape == (8, 512)
assert h_random.shape == (8, 512)

print("Backbone feature extraction passed!")


Pretrained features: torch.Size([8, 512])
Random features: torch.Size([8, 512])
Backbone feature extraction passed!


In [ ]:
# 提取特征
import numpy as np
import torch
from tqdm.auto import tqdm

@torch.no_grad()
def extract_features(backbone, loader, device):
    backbone.eval()

    all_features = []
    all_labels = []

    for images, labels in tqdm(loader, desc="Extracting features"):
        images = images.to(device)

        features = backbone(images)

        all_features.append(features.cpu())
        all_labels.append(labels.cpu())

    X = torch.cat(all_features, dim=0)
    y = torch.cat(all_labels, dim=0)

    return X, y


In [6]:

# MoCo-pretrained features
X_train_moco, y_train = extract_features(
    pretrained_backbone, train_loader, device
)

X_test_moco, y_test = extract_features(
    pretrained_backbone, test_loader, device
)

# Random-backbone features
X_train_random, y_train_random = extract_features(
    random_backbone, train_loader, device
)

X_test_random, y_test_random = extract_features(
    random_backbone, test_loader, device
)

assert torch.equal(y_train, y_train_random)
assert torch.equal(y_test, y_test_random)

print("MoCo train:", X_train_moco.shape)
print("MoCo test:", X_test_moco.shape)
print("Random train:", X_train_random.shape)
print("Random test:", X_test_random.shape)


Extracting features:   0%|          | 0/391 [00:00<?, ?it/s]

Extracting features:   0%|          | 0/79 [00:00<?, ?it/s]

Extracting features:   0%|          | 0/391 [00:00<?, ?it/s]

Extracting features:   0%|          | 0/79 [00:00<?, ?it/s]

MoCo train: torch.Size([50000, 512])
MoCo test: torch.Size([10000, 512])
Random train: torch.Size([50000, 512])
Random test: torch.Size([10000, 512])


In [7]:
# 训练Linear Probe
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

def train_linear_probe(
    X_train,
    y_train,
    X_test,
    y_test,
    device,
    epochs=30,
    lr=0.01,
    batch_size=256,
    seed=42
):
    torch.manual_seed(seed)

    # Only this layer is trainable
    classifier = nn.Linear(
        X_train.shape[1], 10
    ).to(device)

    optimizer = torch.optim.Adam(
        classifier.parameters(),
        lr=lr
    )

    probe_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    X_test_device = X_test.to(device)
    y_test_device = y_test.to(device)

    history = {
        "train_loss": [],
        "train_acc": [],
        "test_acc": []
    }

    for epoch in range(epochs):
        classifier.train()

        total_loss = 0.0
        correct = 0
        total = 0

        for features, labels in probe_loader:
            features = features.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)

            logits = classifier(features)
            loss = F.cross_entropy(logits, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * labels.size(0)

            predictions = logits.argmax(dim=1)
            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

        classifier.eval()

        with torch.no_grad():
            test_logits = classifier(X_test_device)
            test_predictions = test_logits.argmax(dim=1)

            test_acc = (
                test_predictions == y_test_device
            ).float().mean().item()

        train_loss = total_loss / total
        train_acc = correct / total

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_acc)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(
                f"Epoch {epoch+1:02d}/{epochs} | "
                f"Loss: {train_loss:.4f} | "
                f"Train Acc: {train_acc:.4f} | "
                f"Test Acc: {test_acc:.4f}"
            )

    return classifier, history


In [8]:
# MoCo Linear Probe
moco_classifier, moco_history = train_linear_probe(
    X_train=X_train_moco,
    y_train=y_train,
    X_test=X_test_moco,
    y_test=y_test,
    device=device,
    epochs=30,
    lr=0.01
)


Epoch 01/30 | Loss: 1.6301 | Train Acc: 0.4315 | Test Acc: 0.4701
Epoch 05/30 | Loss: 1.3539 | Train Acc: 0.5197 | Test Acc: 0.4801
Epoch 10/30 | Loss: 1.3030 | Train Acc: 0.5373 | Test Acc: 0.5242
Epoch 15/30 | Loss: 1.2700 | Train Acc: 0.5501 | Test Acc: 0.5334
Epoch 20/30 | Loss: 1.2730 | Train Acc: 0.5508 | Test Acc: 0.5302
Epoch 25/30 | Loss: 1.2656 | Train Acc: 0.5548 | Test Acc: 0.5390
Epoch 30/30 | Loss: 1.2733 | Train Acc: 0.5539 | Test Acc: 0.5571


In [9]:
# Random Baseline

random_classifier, random_history = train_linear_probe(
    X_train=X_train_random,
    y_train=y_train,
    X_test=X_test_random,
    y_test=y_test,
    device=device,
    epochs=30,
    lr=0.01
)

moco_acc = moco_history["test_acc"][-1]
random_acc = random_history["test_acc"][-1]

print("\n===== Linear Probe Comparison =====")
print(f"MoCo Accuracy:   {moco_acc:.2%}")
print(f"Random Accuracy: {random_acc:.2%}")
print(
    f"Improvement:     "
    f"{(moco_acc - random_acc) * 100:+.2f} pp"
)


Epoch 01/30 | Loss: 2.5639 | Train Acc: 0.1881 | Test Acc: 0.2622
Epoch 05/30 | Loss: 1.9373 | Train Acc: 0.3103 | Test Acc: 0.3159
Epoch 10/30 | Loss: 1.8693 | Train Acc: 0.3416 | Test Acc: 0.3087
Epoch 15/30 | Loss: 1.8041 | Train Acc: 0.3601 | Test Acc: 0.3473
Epoch 20/30 | Loss: 1.7773 | Train Acc: 0.3721 | Test Acc: 0.3848
Epoch 25/30 | Loss: 1.7618 | Train Acc: 0.3787 | Test Acc: 0.4188
Epoch 30/30 | Loss: 1.7775 | Train Acc: 0.3757 | Test Acc: 0.3367

===== Linear Probe Comparison =====
MoCo Accuracy:   55.71%
Random Accuracy: 33.67%
Improvement:     +22.04 pp



## Preliminary Linear Probing Results

We evaluated a CIFAR-adapted ResNet-18 backbone after
5 epochs of MoCo self-supervised pretraining.

A frozen pretrained backbone achieved **55.71% test
accuracy**, compared with **33.67%** for a randomly
initialized frozen backbone, giving a preliminary
improvement of **22.04 percentage points**.

However, the random baseline exhibited considerable
optimization instability, reaching 41.88% accuracy
at epoch 25 before declining to 33.67% at epoch 30.

These findings suggest that MoCo pretraining improves
linear separability under the current experimental
configuration. A validation-based evaluation protocol
and multiple random seeds are needed before making
stronger claims.
